In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
import matplotlib.pyplot as plt
import seaborn as sns
import re
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
sns.set_palette("gray", 30)
sns.set_style("darkgrid")
cmap = "Greys"

# Steam game Recommender System - Content-Based

## Outlines:
- ?

# 

# Import Dataset

In [ ]:
steam = pd.read_csv(
    "/kaggle/input/steam-store-games/steam.csv")
steam_description = pd.read_csv(
    "/kaggle/input/steam-store-games/steam_description_data.csv")
steam_media_data = pd.read_csv(
    "/kaggle/input/steam-store-games/steam_media_data.csv")

In [ ]:
steam

In [ ]:
steam_description

In [ ]:
steam_media_data 

# Data Health

In [ ]:
class HealthCheck:
    def __init__(self, cmap="Greys"):
        self.cmap = cmap
    def msno_bar(self, data):
        return msno.bar(data)
    
    def msno_matrix(self, data):
        return msno.matrix(data)
    def msno_heatmap(self, data):
        return msno.heatmap(data)
    
    def dendrogram(self, data):

        max_text = 50
        print("-"*max_text)
        print("\033[1mDendrogram\033[0m")
        print("-"*max_text)
        text = "dendrogram The dendrogram allows you to more fully correlate variable completion, revealing trends deeper than the pairwise ones visible in the correlation heatmap:"

        lines = [text[i:i+max_text] for i in range(0, len(text), max_text)]
        for line in lines:
            print(line)
        print("-"*max_text)
        return msno.dendrogram(data)
    
    def stat(self, data):
        return data.describe().T.round(2)

    def dtype_check(self, data):
        ax = data.dtypes.value_counts().plot(kind='bar', color="grey")
        # Add numbers on top of each bar
        for p in ax.patches:
            ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='center', xytext=(0, 9), textcoords='offset points')
        plt.show()
        
    def summary_tbl(self, data: pd.DataFrame) -> pd.DataFrame:
        def get_least_most(column):
            value_counts = data[column].value_counts()
            if len(value_counts) == 0:
                return None, None
            least = value_counts.index[-1] if len(value_counts) > 1 else value_counts.index[0]
            most = value_counts.index[0]
            return least, most
        
        result = pd.DataFrame({
            "dtype": data.dtypes.values,
            "cnt_missing": data.isna().sum(),
            "ratio_missing": (data.isna().sum() / data.shape[0]) * 100,
            "n_unique": data.nunique(),
            "ratio_unique": (data.nunique() / data.shape[0]) * 100,
            "least": [get_least_most(col)[0] for col in data.columns],
            "most": [get_least_most(col)[1] for col in data.columns],
        }, index=data.columns)

        memory_usage = data.memory_usage(deep=True).sum() / 1024**2
        result['memory_usage_mb'] = memory_usage
        return result

    def summary(self, data: pd.DataFrame) -> str:
        summary_df = self.summary_tbl(data=data)
        columns = summary_df.index
        
        # Existing least and most logic
        for column in columns:
            # ... existing logic for determining least and most
            
            summary_df.loc[column, "least"] = least
            summary_df.loc[column, "most"] = most
            
        memory_usage = data.memory_usage(deep=True).sum() / 1024**2
        print(summary_df.round(2).to_markdown())
        print("\n")
        print(f"Row: {data.shape[0]}, Column: {data.shape[1]}, Memory Usage: {memory_usage:.2f} MB")


    def display(self, data: pd.DataFrame):
        display(HTML(data.to_html()))

    def missing_check(self, data):
        return pd.DataFrame({'Missing Values': data.isna().sum().sort_values(ascending=False),
                             'Percentage Missing Values': (data.isna().sum().sort_values(ascending=False)) / (data.shape[0]) * (100)}).style.background_gradient(cmap=self.cmap)
    
    def get_missing(self, data):
        return pd.DataFrame({'Missing Values': data.isna().sum().sort_values(ascending=False),
                             'Percentage Missing Values': (data.isna().sum().sort_values(ascending=False)) / (data.shape[0]) * (100)})

In [ ]:
HC = HealthCheck()

In [ ]:
HC.dtype_check(data=steam)

In [ ]:
HC.summary_tbl(steam)

In [ ]:
HC.summary_tbl(steam_description)

In [ ]:
HC.missing_check(steam_media_data)

In [ ]:
HC.missing_check(steam_description)

In [ ]:
HC.missing_check(steam_media_data)

# Data Processing

## 1. Downcasting
- **Objective**: downcasting the dataframes to reduce the amount of storage 
- Benefits of downcasting in pandas (data manipulation library):
    - Reduces memory usage: By converting data types to smaller sizes based on their actual values (e.g., from float64 to int8).
    - Improves performance: Smaller data types can lead to faster operations in some cases.
    
- Here are the different subtypes you can use:
    - `int8 / uint8` : consumes 1 byte of memory, range between -128/127 or 0/255
    - `bool` : consumes 1 byte, true or false
    - `float16 / int16 / uint16`: consumes 2 bytes of memory, range between -32768 and 32767 or 0/65535
    - `float32 / int32 / uint32` : consumes 4 bytes of memory, range between -2147483648 and 2147483647
    - `float64 / int64 / uint64`: consumes 8 bytes of memory 
    
If one of your column has values between 1 and 10 for example, you will reduce the size of that column from 8 bytes per row to 1 byte, which is more than 85% memory saving on that column!

In [ ]:
def downcast(data):
    cols = data.dtypes.index.tolist()
    types = data.dtypes.values.tolist()

    for i, t in enumerate(types):
        if pd.api.types.is_numeric_dtype(t):  # Check for numeric dtype
            # Handling non-finite values
            data[cols[i]] = data[cols[i]].replace([np.inf, -np.inf], np.nan)
            # Replace NaN with a specific value or drop rows
            data[cols[i]].fillna(value=0, inplace=True)  # Example: Replace NaN with 0
            # Downcasting
            if data[cols[i]].min() > np.iinfo(np.int8).min and data[cols[i]].max() < np.iinfo(np.int8).max:
                data[cols[i]] = data[cols[i]].astype(np.int8)
            elif data[cols[i]].min() > np.iinfo(np.int16).min and data[cols[i]].max() < np.iinfo(np.int16).max:
                data[cols[i]] = data[cols[i]].astype(np.int16)
            elif data[cols[i]].min() > np.iinfo(np.int32).min and data[cols[i]].max() < np.iinfo(np.int32).max:
                data[cols[i]] = data[cols[i]].astype(np.int32)
            else:
                data[cols[i]] = data[cols[i]].astype(np.float64)  

        elif t == object:  
            if cols[i] == 'date':
                data[cols[i]] = pd.to_datetime(data[cols[i]], format='%Y-%m-%d')
            else:
                data[cols[i]] = data[cols[i]].astype('category')
    return data

In [ ]:
def display_variable_memory_usage(dfs_dict, title="Variable Memory Usage"):
    results = []
    for name, df in dfs_dict.items():
        before_memory_usage = df.memory_usage(deep=True).sum() / 1024**2
        after_memory_usage = downcast(df).memory_usage(deep=True).sum() / 1024**2
        memory_usage_reduction_percent = ((before_memory_usage - after_memory_usage) / before_memory_usage) * 100
        results.append({
            "name": name,
            "before_memory_usage": before_memory_usage,
            "after_memory_usage": after_memory_usage,
            "reduction_percent": memory_usage_reduction_percent
        })
    df = pd.DataFrame(results)

    fig = go.Figure(data=[
        go.Bar(
            x=df["name"],
            y=df["before_memory_usage"],
            name="Before Downcasting",
            text=df["before_memory_usage"].apply("{:.2f} MB".format),
            textposition="outside"
        ),
        go.Bar(
            x=df["name"] ,
            y=df["after_memory_usage"],
            name="After Downcasting",
            text=df["after_memory_usage"].apply("{:.2f} MB".format) + df["reduction_percent"].apply("<br> ({:.2f}% reduction)".format) ,
            textposition="outside"
        )
    ])

    fig.update_layout(
        title=title,
        xaxis_title="DataFrame",
        yaxis_title="Memory Usage (MB)",
        barmode="group"
    )

    return fig.show()

In [ ]:
dfs_dict = {'steam':steam,
           'steam description':steam_description,
           'steam media':steam_media_data,}

In [ ]:
display_variable_memory_usage(dfs_dict=dfs_dict,
                              title="<b>Variable Memory Usage</b><br></br>how much effect downcasting has had on the memory usage of DataFrames")

## 2. Handling Missing Values

In [ ]:
HC.missing_check(steam)

In [ ]:
steam.dropna(inplace=True)

In [ ]:
HC.missing_check(steam)

## 3. Handling Duplicates

In [ ]:
duplicate_rows = steam[steam.duplicated(keep=False)]
duplicate_rows

# Encode Categorical Variables
Based on `Data Health`:
1. **Use One-Hot Encoding with Multi-label Binarizer**:
    - For `categories`, `genres`, `platforms`, and `steamspy_tags` since they can contain multiple values per entry, One-Hot Encoding captures the presence or absence of each category/genre without implying any order or relationship between them.
2. **Use Label Encoding**:
    - For `developer` and `publisher`, due to their high cardinality and the need for efficient representation in clustering, Label Encoding simplifies the features into numerical format, allowing distance-based algorithms to operate effectively without excessive dimensionality.

In [ ]:
HC.summary_tbl(steam)

In [ ]:
steam_approx = steam.copy()

In [ ]:
# Function to split `categories`, `genres`, `platforms`, and `steamspy_tags`
def split_categories(column):
    return column.str.split(';')

# Apply the function to split the categories and genres
steam_approx['categories'] = split_categories(steam_approx['categories'])
steam_approx['genres'] = split_categories(steam_approx['genres'])
steam_approx['platforms'] = split_categories(steam_approx['platforms'])
steam_approx['steamspy_tags'] = split_categories(steam_approx['steamspy_tags'])
steam_approx

## 1. One-Hot Encoding with Multi-label Binarizer
**One-Hot Encoding**: For categorical features like `categories`, `genres`, `platforms`, and `steamspy_tags`, use one-hot encoding.
### Why Use One-Hot Encoding with Multi-label Binarizer?
- impact the quality of the clusters produced.  how using One-Hot Encoding for multi-label categorical variables (`categories`, `genres`, `platforms`, and `steamspy_tags`) can lead to better, well-separated clusters

### Nature of the Data

- **Multi-Label Categorical Variables**: The `categories`, `genres`, `platforms`, and `steamspy_tags` columns contain multi-label categorical data, meaning that each entry can contain multiple values separated by a delimiter (e.g., semicolon `;`). For instance, a game might belong to multiple categories like "Single-player;Co-op" or "Adventure;Indie."
- **Example**:
    - `categories`: "Single-player;Co-op"
    - `genres`: "Adventure;Casual;Indie"

### One-Hot Encoding Benefits

- **Independent Features**: One-Hot Encoding allows you to treat each category or genre as a separate binary feature (0 or 1). This method effectively creates new columns for each possible category or genre, indicating whether a particular game belongs to that category or genre.
- **Avoiding Ordinality**: Since categories and genres are nominal variables (no intrinsic order), One-Hot Encoding is appropriate. It avoids introducing any false sense of ordinality that could occur if you were to use Label Encoding, where categories are represented as integers.

### Multi-label Binarizer

- **Handling Multiple Labels**: A Multi-label Binarizer is specifically designed to handle cases where multiple labels can apply to a single sample. It will create a binary column for each unique label across all entries, allowing you to represent the multi-label nature of your `categories`, `genres`, `platforms`, and `steamspy_tags` columns effectively.

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

### Use MultiLabelBinarizer for `categories`, `genres`, `platforms`, and `steamspy_tags`

In [ ]:
# Step 1: Encode categories
mlb_categories = MultiLabelBinarizer()
categories_encoded = mlb_categories.fit_transform(steam_approx['categories'])
categories_df = pd.DataFrame(categories_encoded, columns=mlb_categories.classes_)

# Step 2: Encode genres
mlb_genres = MultiLabelBinarizer()
genres_encoded = mlb_genres.fit_transform(steam_approx['genres'])
genres_df = pd.DataFrame(genres_encoded, columns=mlb_genres.classes_)

# Step 3: Encode platforms
mlb_platforms = MultiLabelBinarizer()
platforms_encoded = mlb_platforms.fit_transform(steam_approx['platforms'])
platforms_df = pd.DataFrame(platforms_encoded, columns=mlb_platforms.classes_)

# Step 4: Encode steamspy_tags
mlb_steamspy_tags = MultiLabelBinarizer()
steamspy_tags_encoded = mlb_steamspy_tags.fit_transform(steam_approx['steamspy_tags'])
steamspy_tags_df = pd.DataFrame(steamspy_tags_encoded, columns=mlb_steamspy_tags.classes_)

### Combining Encoded DataFrames

In [ ]:
steam_approx_encoded_combined = pd.concat([steam_approx, categories_df, genres_df, platforms_df, steamspy_tags_df], axis=1)
steam_approx_encoded_combined.dropna(inplace=True)
steam_approx_encoded_combined

## 2. Label Encoding

### Why Use Label Encoding for Categorical Features?
- Based on `Data Health`, the `developer` or `publisher` columns have a high number of unique values (`n_unique`)
- impact the quality of the clusters produced.  how using Label Encoding for high cardinality categorical variables (`developer` and `publisher`) can lead to better, well-separated clusters
- **High Cardinality**: The `developer` column has **17,102** unique entries, and the `publisher` column has **14,353** unique entries. Such high cardinality means that using them as categorical variables directly can lead to inefficiencies and complications in model training.
- **Numerical Format**: Label encoding transforms these categories into integers (e.g., `developer` entries might be converted from strings like "Developer A" and "Developer B" to integers 0 and 1, respectively). This makes it easier for machine learning models to interpret and process the data.

### Nature of the Data

- **High Cardinality Categorical Variables**: The `developer` and `publisher` columns contain a high number of unique values (17,102 for `developer` and 14,353 for `publisher`). Each entry corresponds to a distinct entity (a game developer or publisher).
- **Example**:
    - `developer`: "Choice of Games"
    - `publisher`: "Big Fish Games"

### Label Encoding Benefits

- **Simplification**: Label Encoding converts categories into integers, which simplifies the representation of high-cardinality categorical variables without the explosion of dimensions that would occur with One-Hot Encoding.
- **Efficient Representation**: Given the high cardinality, using Label Encoding keeps the dataset more manageable and computationally efficient while still allowing distance-based clustering algorithms to interpret the differences among developers and publishers.
- **No Intrinsic Ordinality Concern**: While label encoding does introduce integer values, in the context of clustering, the model treats these integers as distinct points without assuming a ranking among developers or publishers.

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
developer_encoder = LabelEncoder()
publisher_encoder = LabelEncoder()

In [ ]:
# Fit and transform the 'developer' column
steam_approx_encoded_combined['developer'] = developer_encoder.fit_transform(steam_approx_encoded_combined['developer'])

# Fit and transform the 'publisher' column
steam_approx_encoded_combined['publisher'] = publisher_encoder.fit_transform(steam_approx_encoded_combined['publisher'])
steam_approx_encoded_combined

# Feature Extraction 

## 1. Extracting Features on `release_date` column
You can extract relevant components from the `release_date`, such as:

- **Year**: Extract the year of release (e.g., 2020, 2021). This can provide insight into trends over time.
- **Month**: Extract the month of release (1-12). This can help identify seasonal trends in game releases.
- **Day**: Extract the day of the month (1-31), although this is usually less relevant for clustering."

In [ ]:
# Convert 'release_date' is in datetime format
steam_approx_encoded_combined['release_date'] = pd.to_datetime(steam_approx_encoded_combined['release_date'])

# Extracting year, month, and day
steam_approx_encoded_combined['release_year'] = steam_approx_encoded_combined['release_date'].dt.year
steam_approx_encoded_combined['release_month'] = steam_approx_encoded_combined['release_date'].dt.month
steam_approx_encoded_combined['release_day'] = steam_approx_encoded_combined['release_date'].dt.day

In [ ]:
steam_approx_encoded_combined

# Correlation Analysis

Before we proceed to KMeans clustering, it's essential to check the correlation between features in our dataset. The presence of **multicollinearity**, where **features are highly correlated**, can potentially affect the clustering process by not allowing the model to learn the actual underlying patterns in the data, as the features do not provide unique information. This could lead to clusters that are not well-separated and meaningful.
steam_approx_encoded_combined

In [ ]:
def correlation(df: pd.DataFrame, method: str = 'pearson', display_text: bool = False,):
    # Select only numeric columns
    numeric_df = df.select_dtypes(include=['number'])
    
    # Calculate the correlation matrix
    correlations = numeric_df.corr(method=method).abs()  # Absolute correlation values
    
    # Plot the heatmap
    plt.figure(figsize=(12, 9))
    sns.heatmap(correlations, cmap=cmap, annot=display_text, fmt='.2f' if display_text else None, linewidths=0.5)
    plt.title(f'Correlation Heatmap\n Method: {method.capitalize()}')
    plt.show()

In [ ]:
corr_cols = ['english',
 'developer',
 'publisher',
 'required_age',
 'achievements',
 'positive_ratings',
 'negative_ratings',
 'average_playtime',
 'median_playtime',
 'price','release_year', 'release_month', 'release_day']

In [ ]:
correlation(steam_approx_encoded_combined[corr_cols], display_text=True)

If we identify multicollinearity, we can utilize dimensionality reduction techniques like PCA. These techniques help in neutralizing the effect of multicollinearity by transforming the correlated features into a new set of uncorrelated variables, preserving most of the original data's variance. This step not only enhances the quality of clusters formed but also makes the clustering process more computationally efficient.

In [ ]:
steam.columns

### Inference:
- `negative_ratings` and `positive_ratings`
- `median_playtime` and `average_playtime`

These high correlations indicate that these variables move closely together, implying a degree of multicollinearity.

# Feature Scaling

meaning transforming the features to have a mean of 0 and a standard deviation of 1. However, not all features require scaling. 

Before we move forward with the clustering and dimensionality reduction, it's imperative to scale our features. This step holds significant importance, especially in the context of distance-based algorithms like K-means and dimensionality reduction methods like PCA. Here's why:

- **For K-means Clustering**: K-means relies heavily on the concept of '**distance**' between data points to form clusters. When features are not on a similar scale, features with larger values can disproportionately influence the clustering outcome, potentially leading to incorrect groupings.
- **For PCA**: PCA aims to find the directions where the data varies the most. When features are not scaled, those with larger values might dominate these components, not accurately reflecting the underlying patterns in the data.

I will proceed to scale the other features in the dataset to prepare it for PCA and K-means clustering.

In [ ]:
from sklearn.preprocessing import StandardScaler

## Select Numerical columns

In [ ]:
numerical_columns = steam_approx_encoded_combined.select_dtypes(include=['number']).columns.to_list()
excluded_cols = ['appid']

In [ ]:
columns_to_scale= [col for col in numerical_columns if col not in excluded_cols]
columns_to_scale

In [ ]:
scaler = StandardScaler()

In [ ]:
steam_numerical_scaled = scaler.fit_transform(steam_approx_encoded_combined[columns_to_scale])
steam_numerical_scaled_df = pd.DataFrame(steam_numerical_scaled, columns=steam_approx_encoded_combined[columns_to_scale].columns)
steam_numerical_scaled_df

# Dimensionality Reduction

## Why We Need Dimensionality Reduction?
If we identify multicollinearity, we can utilize dimensionality reduction techniques like PCA. These techniques help in neutralizing the effect of multicollinearity by transforming the correlated features into a new set of uncorrelated variables, preserving most of the original data's variance. This step not only enhances the quality of clusters formed but also makes the clustering process more computationally efficient.

- **Multicollinearity Detected**: In the previous steps, we identified that our dataset contains multicollinear features. Dimensionality reduction can help us remove redundant information and alleviate the multicollinearity issue.
- **Better Clustering with K-means**: Since K-means is a distance-based algorithm, having a large number of features can sometimes dilute the meaningful underlying patterns in the data. By reducing the dimensionality, we can help K-means to find more compact and well-separated clusters.
- **Noise Reduction**: By focusing only on the most important features, we can potentially remove noise in the data, leading to more accurate and stable clusters.
- **Enhanced Visualization**: In the context of customer segmentation, being able to visualize customer groups in two or three dimensions can provide intuitive insights. Dimensionality reduction techniques can facilitate this by reducing the data to a few principal components which can be plotted easily.
- **Improved Computational Efficiency**: Reducing the number of features can speed up the computation time during the modeling process, making our clustering algorithm more efficient.

## Which Dimensionality Reduction Method?

In this step, we are considering the application of dimensionality reduction techniques to simplify our data while retaining the essential information. Among various methods such as KernelPCA, ICA, ISOMAP, TSNE, and UMAP, I am starting with **PCA (Principal Component Analysis)**. Here's why:

PCA is an excellent starting point because it works well in capturing linear relationships in the data, which is particularly relevant given the multicollinearity we identified in our dataset. It allows us to reduce the number of features in our dataset while still retaining a significant amount of the information, thus making our clustering analysis potentially more accurate and interpretable. Moreover, it is computationally efficient, which means it won't significantly increase the processing time.

However, it's essential to note that we are keeping our options open. After applying PCA, if we find that the first few components do not capture a significant amount of variance, indicating a loss of vital information, we might consider exploring other non-linear methods. These methods can potentially provide a more nuanced approach to dimensionality reduction, capturing complex patterns that PCA might miss, albeit at the cost of increased computational time and complexity.

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
def plot_scree_cumulative(data, where_n_component, display_percent_every_n=3):
    # Check if the data is a NumPy array, convert it to DataFrame if necessary
    if isinstance(data, np.ndarray):
        data = pd.DataFrame(data)

    # Apply PCA
    pca = PCA()
    df_pca = pca.fit_transform(data)  
    
    explain_ratio = pca.explained_variance_ratio_
    explain_ratio_cum = np.cumsum(explain_ratio)  # Cumulative 
    
    # DataFrame for the first two principal components
    df_pca_components = pd.DataFrame(df_pca[:, :2], columns=['X', 'Y'])
    
    # Concatenate original data with PCA components
    data_with_pca = pd.concat([data.reset_index(drop=True), df_pca_components], axis=1)

    fig = go.Figure()

    # Add cumulative variance line
    fig.add_trace(go.Scatter(x=list(range(1, len(explain_ratio) + 1)), y=explain_ratio_cum,
                             mode='lines', name='Cumulative', line=dict(color='firebrick')))
    
    # Change individual variance line to a bar plot
    fig.add_trace(go.Bar(x=list(range(1, len(explain_ratio) + 1)), y=explain_ratio,
                         name='Individual', marker_color='royalblue'))

    # Add annotations for individual and cumulative variance
    for x, ex_ratio, ex_ratio_cum in zip(range(1, len(explain_ratio) + 1), explain_ratio, explain_ratio_cum):
        if x % display_percent_every_n == 0:
            ex_ratio_label = f'{ex_ratio * 100:.0f}%'
            fig.add_annotation(x=x, y=ex_ratio, text=ex_ratio_label,
                               xshift=5, yshift=5, showarrow=False, font=dict(size=10))
            ex_ratio_cum_label = f'{ex_ratio_cum * 100:.0f}%'
            fig.add_annotation(x=x, y=ex_ratio_cum, text=ex_ratio_cum_label,
                               xshift=5, yshift=5, showarrow=False, font=dict(size=10))

    # Mark the "elbow" point
    fig.add_shape(type="line", x0=where_n_component, y0=0, x1=where_n_component, y1=1,
                  line=dict(color="brown", width=1, dash="dash"))
    fig.add_shape(type="line", x0=1, y0=0.01, x1=len(explain_ratio), y1=0.05,
                  line=dict(color="brown", width=1, dash="dash"))
    fig.add_annotation(x=where_n_component, y=0.2, text=f'Variance explained = {(explain_ratio_cum[where_n_component] * 100):.2f}%',
                       showarrow=False, font=dict(size=12), xshift=5, yshift=5)

    # Set x-axis limits
    fig.update_xaxes(range=[1, len(explain_ratio)])

    # Update layout
    fig.update_layout(title_text=f"<b>Scree plot</b> with cumulative variation plot <br><b>criterion</b>: looks for the “elbow” in the curve<br><b>choose:</b> n_components = {where_n_component}",
                      xaxis_title="Number of components",
                      yaxis_title="Variance explained",
                      legend=dict(x=0.02, y=0.98, bgcolor='rgba(255, 255, 255, 0.5)', bordercolor='rgba(0, 0, 0, 0.5)'))
    
    # Show the figure
    fig.show()
    return data_with_pca

In [ ]:
def plot_scree_cumulative(data, where_n_component, display_percent_every_n=3):
    # Check if the data is a NumPy array, convert it to DataFrame if necessary
    if isinstance(data, np.ndarray):
        data = pd.DataFrame(data)

    # Apply PCA
    pca = PCA()
    df_pca = pca.fit_transform(data)  
    
    explain_ratio = pca.explained_variance_ratio_
    explain_ratio_cum = np.cumsum(explain_ratio)  # Cumulative 
    
    # DataFrame for the first two principal components
    df_pca_components = pd.DataFrame(df_pca[:, :2], columns=['X', 'Y'])
    
    # Concatenate original data with PCA components
    data_with_pca = pd.concat([data.reset_index(drop=True), df_pca_components], axis=1)

    fig = go.Figure()

    # Add cumulative variance line
    fig.add_trace(go.Scatter(x=list(range(1, len(explain_ratio) + 1)), y=explain_ratio_cum,
                             mode='lines', name='Cumulative', line=dict(color='firebrick')))
    
    # Change individual variance line to a bar plot
    fig.add_trace(go.Bar(x=list(range(1, len(explain_ratio) + 1)), y=explain_ratio,
                         name='Individual', marker_color='royalblue'))

    # Add annotations for individual and cumulative variance
    for x, ex_ratio, ex_ratio_cum in zip(range(1, len(explain_ratio) + 1), explain_ratio, explain_ratio_cum):
        if x % display_percent_every_n == 0:
            ex_ratio_label = f'{ex_ratio * 100:.0f}%'
            fig.add_annotation(x=x, y=ex_ratio, text=ex_ratio_label,
                               xshift=5, yshift=5, showarrow=False, font=dict(size=10))
            ex_ratio_cum_label = f'{ex_ratio_cum * 100:.0f}%'
            fig.add_annotation(x=x, y=ex_ratio_cum, text=ex_ratio_cum_label,
                               xshift=5, yshift=5, showarrow=False, font=dict(size=10))

    # Mark the "elbow" point
    fig.add_shape(type="line", x0=where_n_component, y0=0, x1=where_n_component, y1=1,
                  line=dict(color="brown", width=1, dash="dash"))
    fig.add_shape(type="line", x0=1, y0=0.01, x1=len(explain_ratio), y1=0.05,
                  line=dict(color="brown", width=1, dash="dash"))
    fig.add_annotation(x=where_n_component, y=0.2, text=f'Variance explained = {(explain_ratio_cum[where_n_component] * 100):.2f}%',
                       showarrow=False, font=dict(size=12), xshift=5, yshift=5)

    # Set x-axis limits
    fig.update_xaxes(range=[1, len(explain_ratio)])

    # Update layout
    fig.update_layout(title_text=f"<b>Scree plot</b> with cumulative variation plot <br><b>criterion</b>: looks for the “elbow” in the curve<br><b>choose:</b> n_components = {where_n_component}",
                      xaxis_title="Number of components",
                      yaxis_title="Variance explained",
                      legend=dict(x=0.02, y=0.98, bgcolor='rgba(255, 255, 255, 0.5)', bordercolor='rgba(0, 0, 0, 0.5)'))
    
    # Show the figure
    fig.show()
    return data_with_pca

In [ ]:
data_with_pca = plot_scree_cumulative(data=steam_numerical_scaled_df, 
                      where_n_component=4, 
                      display_percent_every_n=20)

To choose the optimal number of components, we generally look for a point where adding another component doesn't significantly increase the cumulative explained variance, often referred to as the "**elbow point**" in the curve.

From the plot, we can see that the increase in cumulative variance starts to slow down after the **6th component** (which **captures about 81% of the total variance**).

# KMeans
- **K-Means** is an unsupervised machine learning algorithm that clusters data into a specified number of groups (K) by minimizing the **within-cluster sum-of-squares (WCSS)**, also known as **inertia**. The algorithm iteratively assigns each data point to the nearest centroid, then updates the centroids by calculating the mean of all assigned points. The process repeats until convergence or a stopping criterion is reached.

## Elbow Method

### What is the Elbow Method?

The Elbow Method is a technique for identifying the ideal number of clusters in a dataset. It involves iterating through the data, generating clusters for various values of k. The k-means algorithm calculates the sum of squared distances between each data point and its assigned cluster centroid, known as the **inertia** or **WCSS** score. By plotting the inertia score against the k value, we create a graph that typically exhibits an elbow shape, hence the name "**Elbow Method**". The **elbow point** represents the k-value where the reduction in inertia achieved by increasing k becomes negligible, indicating the optimal stopping point for the number of clusters.

In [ ]:
from sklearn.cluster import KMeans

In [ ]:
wcss = []
k_values = range(1, 11)  # Test for k from 1 to 10

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(steam_numerical_scaled_df)
    wcss.append(kmeans.inertia_)  # WCSS

In [ ]:
# Select the number of clusters (e.g., 3 as an example)
optimal_k = 3

# Plotting the scree plot to find the elbow
plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), wcss, marker='o', linestyle='--')
plt.axvline(x=optimal_k, color='r', linestyle='--', label=f'Optimal K = {optimal_k}')
plt.title('Scree Plot: The Elbow Method')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Within-Cluster Sum of Squares (WCSS)')
plt.grid()
plt.legend()
plt.show()

### Optimal k Value: Elbow Method Insights

The optimal value of k for the KMeans clustering algorithm can be found at the **elbow point**. Using the YellowBrick library for the Elbow method, we observe that the suggested optimal k value is **5**. However, **we don't have a very distinct elbow point in this case**, which is common in real-world data. From the plot, we can see that the inertia continues to decrease significantly up to k=5, indicating that **the optimum value of k could be between 3 and 7**. To choose the best k within this range, we can employ the **silhouette analysis**, another cluster quality evaluation method. Additionally, incorporating business insights can help determine a practical k value.

## Silhouette Method
### What is the Silhouette Method?

The **Silhouette Method** is an approach to find the optimal number of clusters in a dataset by evaluating the consistency within clusters and their separation from other clusters. It computes the **silhouette coefficient for each data point**, which measures how similar a point is to its own cluster compared to other clusters.

---

### What is the Silhouette Coefficient?

To determine the silhouette coefficient for a given point i, follow these steps:

- **Calculate a(i)**: Compute the average distance between point i and all other points within its cluster.
- **Calculate b(i)**: Compute the average distance between point i and all points in the nearest cluster to its own.
- **Compute the silhouette coefficient**, $s(i)$, for point i using the following formula:
    
$$s(i) = \frac{b(i) - a(i)}{\max\{a(i), b(i)\}}$$
    

**Note:** The silhouette coefficient quantifies the similarity of a point to its own cluster (cohesion) relative to its separation from other clusters. This value ranges from -1 to 1, with higher values signifying that the point is well aligned with its cluster and has a low similarity to neighboring clusters.

---

### What is the Silhouette Score?

The **silhouette score** is the **average silhouette coefficient** calculated for all data points in a dataset. It provides an overall assessment of the clustering quality, taking into account both cohesion within clusters and separation between clusters. A higher silhouette score indicates a better clustering configuration.

---

### What are the Advantages of Silhouette Method over the Elbow Method?

- The **Silhouette Method** evaluates cluster quality by considering **both** the **cohesion within clusters** and their **separation** from other clusters. This provides a more comprehensive measure of clustering performance compared to the **Elbow Method**, which only considers the **inertia** (sum of squared distances within clusters).
- The **Silhouette Method** produces a silhouette score that directly quantifies the quality of clustering, making it easier to compare different values of k. In contrast, the **Elbow Method** relies on the subjective interpretation of the elbow point, which can be less reliable in cases where the plot does not show a clear elbow.
- The **Silhouette Method** generates a visual representation of silhouette coefficients for each data point, allowing for easier identification of fluctuations and outliers within clusters. This helps in determining the optimal number of clusters with higher confidence, as opposed to the **Elbow Method**, which relies on visual inspection of the inertia plot.

In [ ]:
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from yellowbrick.cluster import KElbowVisualizer, SilhouetteVisualizer
import matplotlib.gridspec as gridspec

In [ ]:
def silhouette_analysis(df, start_k, stop_k, figsize=(15, 16)):
    """
    Perform Silhouette analysis for a range of k values and visualize the results.
    """

    # Set the size of the figure
    plt.figure(figsize=figsize)

    # Create a grid with (stop_k - start_k + 1) rows and 2 columns
    grid = gridspec.GridSpec(stop_k - start_k + 1, 2)

    # Assign the first plot to the first row and both columns
    first_plot = plt.subplot(grid[0, :])

    # First plot: Silhouette scores for different k values
    sns.set_palette(['darkorange'])

    silhouette_scores = []

    # Iterate through the range of k values
    for k in range(start_k, stop_k + 1):
        km = KMeans(n_clusters=k, init='k-means++', n_init=10, max_iter=100, random_state=0)
        km.fit(df)
        labels = km.predict(df)
        score = silhouette_score(df, labels)
        silhouette_scores.append(score)

    best_k = start_k + silhouette_scores.index(max(silhouette_scores))

    plt.plot(range(start_k, stop_k + 1), silhouette_scores, marker='o')
    plt.xticks(range(start_k, stop_k + 1))
    plt.xlabel('Number of clusters (k)')
    plt.ylabel('Silhouette score')
    plt.title('Average Silhouette Score for Different k Values', fontsize=15)

    # Add the optimal k value text to the plot
    optimal_k_text = f'The k value with the highest Silhouette score is: {best_k}'
    plt.text(10, 0.23, optimal_k_text, fontsize=12, verticalalignment='bottom', 
             horizontalalignment='left', bbox=dict(facecolor='#fcc36d', edgecolor='#ff6200', boxstyle='round, pad=0.5'))
             

    # Second plot (subplot): Silhouette plots for each k value
    colors = sns.color_palette("bright")

    for i in range(start_k, stop_k + 1):    
        km = KMeans(n_clusters=i, init='k-means++', n_init=10, max_iter=100, random_state=0)
        row_idx, col_idx = divmod(i - start_k, 2)

        # Assign the plots to the second, third, and fourth rows
        ax = plt.subplot(grid[row_idx + 1, col_idx])

        visualizer = SilhouetteVisualizer(km, colors=colors, ax=ax)
        visualizer.fit(df)

        # Add the Silhouette score text to the plot
        score = silhouette_score(df, km.labels_)
        ax.text(0.97, 0.02, f'Silhouette Score: {score:.2f}', fontsize=12, \
                ha='right', transform=ax.transAxes, color='red')

        ax.set_title(f'Silhouette Plot for {i} Clusters', fontsize=15)

    plt.tight_layout()
    plt.show()

In [ ]:
silhouette_analysis(steam_numerical_scaled_df, 
                    3, 
                    12, 
                    figsize=(20, 50))


### **Guidelines to Interpret Silhouette Plots and Determine the Optimal K:**

To interpret silhouette plots and identify the optimal number of clusters (( k )), consider the following criteria:

- 1️⃣ **Analyze the Silhouette Plots**:
    - **Silhouette Score Width**:
        - **Wide Widths (closer to +1)**: Indicate that the data points in the cluster are well separated from points in other clusters, suggesting well-defined clusters.
        - **Narrow Widths (closer to -1)**: Show that data points in the cluster are not distinctly separated from other clusters, indicating poorly defined clusters.
    - **Average Silhouette Score**:
        - **High Average Width**: A cluster with a high average silhouette score indicates well-separated clusters.
        - **Low Average Width**: A cluster with a low average silhouette score indicates poor separation between clusters.

---

- 2️⃣ **Uniformity in Cluster Size**:
    
    2.1 **Cluster Thickness**:
    
    - **Uniform Thickness**: Indicates that clusters have a roughly equal number of data points, suggesting a balanced clustering structure.
    - **Variable Thickness**: Signifies an imbalance in the data point distribution across clusters, with some clusters having many data points and others too few.

---

- 3️⃣ **Peaks in Average Silhouette Score**:
    - **Clear Peaks**: A clear peak in the **average** silhouette score plot for a specific ( k ) value indicates this ( k ) might be optimal.

---

- 4️⃣ **Minimize Fluctuations in Silhouette Plot Widths**:
    - **Uniform Widths**: Seek silhouette plots with similar widths across clusters, suggesting a more balanced and optimal clustering.
    - **Variable Widths**: Avoid wide fluctuations in silhouette plot widths, indicating that clusters are not well-defined and may vary in compactness.

---

- 5️⃣ **Optimal Cluster Selection**:
    - **Maximize the Overall Average Silhouette Score**: Choose the ( k ) value that gives the highest average silhouette score across all clusters, indicating well-defined clusters.
    - **Avoid Below-Average Silhouette Scores**: Ensure most clusters have above-average silhouette scores to prevent suboptimal clustering structures.

---

- 6️⃣ **Visual Inspection of Silhouette Plots**:
    - **Consistent Cluster Formation**: Visually inspect the silhouette plots for each ( k ) value to evaluate the consistency and structure of the formed clusters.
    - **Cluster Compactness**: Look for more compact clusters, with data points having silhouette scores closer to +1, indicating better clustering.

### Optimal k Value: Silhouette Method Insights

Based on above guidelines and after carefully considering the silhouette plots, it's clear that choosing **( k = 3 )** is the better option. This choice gives us clusters that are more evenly matched and well-defined, making our clustering solution stronger and more reliable.

## Clusterize

In [ ]:
def clusterize(data, optimal_k):
    # Check if the data is a NumPy array, convert it to DataFrame if necessary
    if isinstance(data, np.ndarray):
        data = pd.DataFrame(data)
    kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42)
    clusters = kmeans.fit_predict(data)
    data['Cluster'] = clusters
    return data

In [ ]:
steam_numerical_scaled_df_clusterized = clusterize(steam_numerical_scaled_df, 3)

In [ ]:
steam_numerical_scaled_df_clusterized

In [ ]:
def cluster(data):
    hover_texts = data.apply(
        lambda row: '<br>'.join([f"{col}: {row[col]}" for col in data.columns]),
        axis=1
    )


    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=data['X'],
        y=data['Y'],
        mode='markers',
        marker=dict(
            size=10,
            color=data['Cluster'], 
            colorscale='Viridis', 
            showscale=True,
            line=dict(width=1, color='black')
        ),
        text=hover_texts, 
        hoverinfo='text', 
        name='Clusters'
    ))

    fig.update_layout(
        title='Clusters in PCA Space',
        xaxis_title='Principal Component 1',
        yaxis_title='Principal Component 2',
        xaxis=dict(
            showgrid=True,  
            gridcolor='LightGray', 
            gridwidth=0.5 
        ),
        yaxis=dict(
            showgrid=True, 
            gridcolor='LightGray', 
            gridwidth=0.5  
        ),
        legend_title='Cluster',
    )

    fig.show()
    

In [ ]:
cluster(steam_numerical_scaled_df_clusterized)